# Notebook 09 — New Probabilistic Models: LSTM-KDE, MDN, ARIMA

Three additional probabilistic forecasting approaches as requested by the professor:

| Model | Method | Key idea |
|---|---|---|
| **LSTM-KDE** | LSTM + Kernel Density Estimation | Non-parametric residual distribution |
| **MDN-GRU** | Mixture Density Network | K=3 Gaussian mixture output |
| **ARIMA(1,1,1)** | Classical SARIMA | Baseline: analytical PI from error variance |

**Metrics computed:** PICP, MPIW, Winkler Score (WIS), CWC  
**Env:** Set `RUN_ENV='local'` for VS Code/lab PC.

In [ ]:
# ── Cell 1: Environment + Imports ────────────────────────────
RUN_ENV  = 'local'
BASE_DIR = r'C:/BMP_Data/'
if RUN_ENV == 'colab':
    from google.colab import drive; drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/BMP_Data/'

import os, warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import KernelDensity
from scipy.stats import norm as scipy_norm
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.tsa.arima.model import ARIMA
    STATSMODELS_OK = True
except ImportError:
    print("Install statsmodels: pip install statsmodels"); STATSMODELS_OK = False

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

SAVE_DIR  = os.path.join(BASE_DIR, 'results_IndianOcean/')
DATA_FILE = os.path.join(BASE_DIR, 'sla_daily_indian_ocean_2021_2023.nc')
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'TF {tf.__version__} | Environment: {RUN_ENV}')

In [ ]:
# ── Cell 2: Shared config + data loader ──────────────────────
LOCATIONS = {
    'Arabian_Sea':   (15.0, 65.0),
    'Bay_of_Bengal': (12.0, 87.0),
    'Andaman_Sea':   (11.0, 95.0),
    'Lakshadweep':   (10.0, 73.0),
    'South_IO':      (-5.0, 75.0),
}

TRAIN_SPLIT = 0.80
CAL_FRAC    = 0.15   # last 15% of train = calibration (for KDE, ARIMA hold-out)
SEQ_LEN     = 30
EPOCHS      = 100
BATCH_SIZE  = 64
LR          = 0.001
PATIENCE    = 20
SEEDS       = [42, 7, 13, 99, 2025]
DROPOUT     = 0.2
ALPHA_WIS   = 0.20
TARGET_PICP = 0.95

ds = xr.open_dataset(DATA_FILE)
times_index = pd.to_datetime(ds['time'].values)

def get_splits(loc_name):
    lat, lon = LOCATIONS[loc_name]
    sla_raw = ds['sla'].sel(latitude=lat, longitude=lon, method='nearest').values.flatten()
    s = pd.Series(sla_raw, index=times_index).interpolate(method='time', limit=14).ffill().bfill()
    sla = s.values
    n_total = len(sla)
    n_train = int(n_total * TRAIN_SPLIT)
    n_test  = n_total - n_train
    n_cal   = int(n_train * CAL_FRAC)
    n_inner = n_train - n_cal
    scaler  = RobustScaler()
    inner_s = scaler.fit_transform(sla[:n_inner].reshape(-1,1)).flatten()
    cal_s   = scaler.transform(sla[n_inner:n_train].reshape(-1,1)).flatten()
    test_s  = scaler.transform(sla[n_train:].reshape(-1,1)).flatten()
    combined_ic   = np.concatenate([inner_s, cal_s])
    combined_full = np.concatenate([scaler.transform(sla[:n_train].reshape(-1,1)).flatten(), test_s])
    X_tr, y_tr = [], []
    for i in range(len(inner_s)-SEQ_LEN):
        X_tr.append(inner_s[i:i+SEQ_LEN]); y_tr.append(inner_s[i+SEQ_LEN])
    X_tr = np.array(X_tr)[..., np.newaxis]; y_tr = np.array(y_tr)
    X_cal = np.array([combined_ic[n_inner-SEQ_LEN+i:n_inner+i] for i in range(n_cal)])[..., np.newaxis]
    y_cal = np.array([combined_ic[n_inner+i] for i in range(n_cal)])
    X_te  = np.array([combined_full[n_train-SEQ_LEN+i:n_train+i] for i in range(n_test)])[..., np.newaxis]
    y_te  = np.array([combined_full[n_train+i] for i in range(n_test)])
    return dict(X_tr=X_tr, y_tr=y_tr, X_cal=X_cal, y_cal=y_cal, X_te=X_te, y_te=y_te,
                scaler=scaler, sla=sla, n_inner=n_inner, n_cal=n_cal, n_test=n_test,
                n_train=n_train)

def winkler_score(yt, lo, hi, alpha=ALPHA_WIS):
    w = hi-lo; b=(2/alpha)*np.maximum(0,lo-yt); a=(2/alpha)*np.maximum(0,yt-hi)
    return float(np.mean(w+b+a))

def cwc_metric(picp_frac, mpiw, target=TARGET_PICP, eta=50):
    if picp_frac >= target: return mpiw
    return mpiw * np.exp(-eta*(picp_frac - target))

def full_metrics(y_true_m, lo_m, hi_m):
    lo = np.minimum(lo_m, hi_m); hi = np.maximum(lo_m, hi_m)
    picp = np.mean((y_true_m >= lo) & (y_true_m <= hi)) * 100.0
    mpiw = np.mean(hi - lo)
    wis  = winkler_score(y_true_m, lo, hi)
    cwc  = cwc_metric(picp/100, mpiw)
    return picp, mpiw, wis, cwc

print(f"Splits ready. Per location: inner_train≈{int(len(times_index)*TRAIN_SPLIT*(1-CAL_FRAC))-SEQ_LEN} seqs")

In [ ]:
# ── Cell 3: Model 1 — LSTM-KDE ───────────────────────────────
# Architecture: LSTM point forecast + Kernel Density Estimation on residuals
#
# Method (Silverman 1986 + Yang et al. 2020):
#   1. Train LSTM to predict SLA mean (single output, MSE loss)
#   2. Compute residuals on calibration set
#   3. Fit KDE on calibration residuals (Gaussian kernel, bandwidth = Silverman's rule)
#   4. Prediction interval: mean ± KDE quantiles that achieve target PICP
#
# Advantage: Non-parametric, captures skewed/heavy-tailed error distributions.
# Disadvantage: Assumes residual distribution is stationary.

def build_lstm_point(seq_len=SEQ_LEN, dropout=DROPOUT):
    inp = keras.Input(shape=(seq_len,1))
    x   = layers.LSTM(64, activation='tanh')(inp)
    x   = layers.Dropout(dropout)(x)
    x   = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    return keras.Model(inp, layers.Dense(1, activation='linear')(x))

def find_kde_quantiles(residuals_scaled, q_lo=0.05, q_hi=0.95):
    """Find KDE quantile bounds that cover (q_hi - q_lo) fraction of data."""
    h = 1.06 * np.std(residuals_scaled) * len(residuals_scaled)**(-0.2)   # Silverman rule
    kde = KernelDensity(bandwidth=h, kernel='gaussian')
    kde.fit(residuals_scaled.reshape(-1,1))
    # Sample large grid to find empirical quantiles
    xs = np.linspace(residuals_scaled.min()*2, residuals_scaled.max()*2, 5000)
    log_dens = kde.score_samples(xs.reshape(-1,1))
    dens     = np.exp(log_dens); dens /= dens.sum()
    cdf      = np.cumsum(dens)
    q_lo_val = xs[np.searchsorted(cdf, q_lo)]
    q_hi_val = xs[np.searchsorted(cdf, q_hi)]
    return q_lo_val, q_hi_val, kde

lstm_kde_results = []
print("Training LSTM-KDE across 5 locations x 5 seeds...")

for loc_name in LOCATIONS:
    d = get_splits(loc_name)
    scaler = d['scaler']
    y_tr_mse = d['y_tr'].reshape(-1,1)   # single output for MSE
    seed_picps, seed_mpiws, seed_wis, seed_cwc = [], [], [], []

    print(f"\n  {loc_name}:")
    for seed in SEEDS:
        tf.random.set_seed(seed); np.random.seed(seed)
        m = build_lstm_point()
        m.compile(optimizer=keras.optimizers.Adam(LR, clipnorm=1.0), loss='mse')
        m.fit(d['X_tr'], y_tr_mse, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.10,
              callbacks=[keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True, verbose=0),
                         keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=10, min_lr=1e-6, verbose=0)],
              verbose=0)

        # Calibration residuals
        mu_cal_s  = m.predict(d['X_cal'], verbose=0).flatten()
        resid_cal = d['y_cal'] - mu_cal_s   # residuals in scaled space

        # KDE on calibration residuals → find quantile offsets
        q_lo_s, q_hi_s, _ = find_kde_quantiles(resid_cal, q_lo=0.05, q_hi=0.95)

        # Test predictions
        mu_te_s  = m.predict(d['X_te'], verbose=0).flatten()
        lo_te_s  = mu_te_s + q_lo_s
        hi_te_s  = mu_te_s + q_hi_s

        # Inverse transform
        lo_m = scaler.inverse_transform(lo_te_s.reshape(-1,1)).flatten()
        hi_m = scaler.inverse_transform(hi_te_s.reshape(-1,1)).flatten()
        y_m  = scaler.inverse_transform(d['y_te'].reshape(-1,1)).flatten()

        picp, mpiw, wis, cwc = full_metrics(y_m, lo_m, hi_m)
        seed_picps.append(picp); seed_mpiws.append(mpiw)
        seed_wis.append(wis); seed_cwc.append(cwc)
        print(f"    seed={seed:4d} | PICP={picp:5.1f}% | MPIW={mpiw:.5f} | WIS={wis:.5f}")

    avg_picp=np.mean(seed_picps); std_picp=np.std(seed_picps)
    avg_mpiw=np.mean(seed_mpiws); std_mpiw=np.std(seed_mpiws)
    avg_wis=np.mean(seed_wis);   std_wis=np.std(seed_wis)
    avg_cwc=np.mean(seed_cwc);   std_cwc=np.std(seed_cwc)
    lat, lon = LOCATIONS[loc_name]
    lstm_kde_results.append(dict(location=loc_name, lat=lat, lon=lon, model='LSTM-KDE',
        avg_picp=avg_picp, std_picp=std_picp, avg_mpiw=avg_mpiw, std_mpiw=std_mpiw,
        avg_wis=avg_wis, std_wis=std_wis, avg_cwc=avg_cwc, std_cwc=std_cwc,
        n_seeds=len(SEEDS), n_train_seqs=len(d['X_tr']), n_test_pts=len(d['X_te'])))
    print(f"  --> PICP={avg_picp:.1f}±{std_picp:.1f}%  MPIW={avg_mpiw:.5f}  CWC={avg_cwc:.5f}")

df_kde = pd.DataFrame(lstm_kde_results)
df_kde.to_csv(os.path.join(SAVE_DIR, 'results_LSTM_KDE_IO.csv'), index=False)
print(f"\nSaved LSTM-KDE results.")
print(df_kde[['location','avg_picp','std_picp','avg_mpiw','avg_cwc']].to_string(index=False))

In [ ]:
# ── Cell 4: Model 2 — MDN (Mixture Density Network) ─────────
# Architecture: GRU backbone + Mixture of K Gaussians output
# Reference: Bishop (1994) "Mixture density networks"
#
# Output: K mixture components, each with (pi_k, mu_k, sigma_k)
# Loss: Negative log-likelihood of Gaussian mixture
# Intervals: Numerical quantiles from the mixture CDF
#
# Advantage: Can model multi-modal and heteroscedastic uncertainty.
# K=3 components captures complexity without over-parameterising.

K = 3   # Number of mixture components

class MDNLayer(layers.Layer):
    """Output layer for K-component Gaussian Mixture Model."""
    def __init__(self, k, **kwargs):
        super().__init__(**kwargs)
        self.k = k
        self.pi_layer    = layers.Dense(k, activation='softmax',   name='pi')
        self.mu_layer    = layers.Dense(k, activation='linear',    name='mu')
        self.sigma_layer = layers.Dense(k, activation='softplus',  name='sigma')

    def call(self, x):
        pi    = self.pi_layer(x) + 1e-8    # mixture weights (sum to 1)
        mu    = self.mu_layer(x)            # component means
        sigma = self.sigma_layer(x) + 1e-5 # component stds (positive)
        return tf.concat([pi, mu, sigma], axis=-1)

def build_mdn(seq_len=SEQ_LEN, k=K, dropout=DROPOUT):
    inp = keras.Input(shape=(seq_len,1))
    x   = layers.GRU(64, activation='tanh')(inp)
    x   = layers.Dropout(dropout)(x)
    x   = layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    out = MDNLayer(k)(x)
    return keras.Model(inp, out)

def mdn_nll(y_true, y_pred, k=K):
    """Gaussian Mixture NLL loss."""
    true = y_true[:, 0:1]   # single true value (duplicated for compatibility)
    pi    = y_pred[:, :k]
    mu    = y_pred[:, k:2*k]
    sigma = y_pred[:, 2*k:3*k]
    # Log-likelihood of each component
    log_norm = -0.5*tf.square((true - mu)/sigma) - tf.math.log(sigma) - 0.5*tf.math.log(2*np.pi)
    log_pi   = tf.math.log(pi)
    log_mix  = tf.reduce_logsumexp(log_pi + log_norm, axis=1)
    return -tf.reduce_mean(log_mix)

def mdn_quantiles(pi_np, mu_np, sigma_np, q_lo=0.05, q_hi=0.95, n_samples=10000):
    """Monte Carlo quantiles from Gaussian mixture."""
    # Sample components proportionally to mixture weights
    idx  = np.random.choice(pi_np.shape[1], size=(n_samples,), p=pi_np[0]/pi_np[0].sum())
    samp = np.random.normal(mu_np[0, idx], sigma_np[0, idx])
    return float(np.quantile(samp, q_lo)), float(np.quantile(samp, q_hi))

def mdn_predict_intervals(model, X, scaler, q_lo=0.05, q_hi=0.95):
    """Predict MDN intervals in original metres space."""
    raw  = model.predict(X, verbose=0)
    pi   = raw[:, :K]
    mu   = raw[:, K:2*K]
    sig  = raw[:, 2*K:3*K]
    # Normalize pi
    pi   = pi / pi.sum(axis=1, keepdims=True)
    los, his = [], []
    for i in range(len(X)):
        idx  = np.random.choice(K, size=5000, p=pi[i])
        samp = np.random.normal(mu[i, idx], sig[i, idx])
        lo_s = float(np.quantile(samp, q_lo))
        hi_s = float(np.quantile(samp, q_hi))
        lo_m = float(scaler.inverse_transform([[lo_s]])[0,0])
        hi_m = float(scaler.inverse_transform([[hi_s]])[0,0])
        los.append(lo_m); his.append(hi_m)
    return np.array(los), np.array(his)

mdn_results = []
print("Training MDN (Mixture Density Network) across 5 locations x 5 seeds...")

for loc_name in LOCATIONS:
    d = get_splits(loc_name)
    scaler = d['scaler']
    y_dup  = np.hstack([d['y_tr'].reshape(-1,1), d['y_tr'].reshape(-1,1)])
    seed_picps, seed_mpiws, seed_wis, seed_cwc = [], [], [], []
    print(f"\n  {loc_name}:")

    for seed in SEEDS:
        tf.random.set_seed(seed); np.random.seed(seed)
        m = build_mdn()
        # Wrap MDN NLL as lambda for compile
        m.compile(optimizer=keras.optimizers.Adam(LR, clipnorm=1.0),
                  loss=lambda yt, yp: mdn_nll(yt, yp, k=K))
        m.fit(d['X_tr'], y_dup, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_split=0.10,
              callbacks=[keras.callbacks.EarlyStopping(patience=PATIENCE, restore_best_weights=True, verbose=0),
                         keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=10, min_lr=1e-6, verbose=0)],
              verbose=0)

        lo_m, hi_m = mdn_predict_intervals(m, d['X_te'], scaler)
        y_m = scaler.inverse_transform(d['y_te'].reshape(-1,1)).flatten()
        picp, mpiw, wis, cwc = full_metrics(y_m, lo_m, hi_m)
        seed_picps.append(picp); seed_mpiws.append(mpiw)
        seed_wis.append(wis); seed_cwc.append(cwc)
        print(f"    seed={seed:4d} | PICP={picp:5.1f}% | MPIW={mpiw:.5f} | WIS={wis:.5f}")

    avg_picp=np.mean(seed_picps); std_picp=np.std(seed_picps)
    avg_mpiw=np.mean(seed_mpiws); std_mpiw=np.std(seed_mpiws)
    avg_wis=np.mean(seed_wis);   std_wis=np.std(seed_wis)
    avg_cwc=np.mean(seed_cwc);   std_cwc=np.std(seed_cwc)
    lat, lon = LOCATIONS[loc_name]
    mdn_results.append(dict(location=loc_name, lat=lat, lon=lon, model='MDN-GRU',
        avg_picp=avg_picp, std_picp=std_picp, avg_mpiw=avg_mpiw, std_mpiw=std_mpiw,
        avg_wis=avg_wis, std_wis=std_wis, avg_cwc=avg_cwc, std_cwc=std_cwc,
        n_seeds=len(SEEDS), n_train_seqs=len(d['X_tr']), n_test_pts=len(d['X_te'])))
    print(f"  --> PICP={avg_picp:.1f}±{std_picp:.1f}%  MPIW={avg_mpiw:.5f}  CWC={avg_cwc:.5f}")

df_mdn = pd.DataFrame(mdn_results)
df_mdn.to_csv(os.path.join(SAVE_DIR, 'results_MDN_IO.csv'), index=False)
print(f"\nSaved MDN results.")

In [ ]:
# ── Cell 5: Model 3 — SARIMA with Prediction Intervals ───────
# Method: SARIMA(p,d,q) + seasonal component for daily SLA
# Reference: Box & Jenkins (1976), Hamilton (1994)
#
# Prediction intervals derived from SARIMA forecast error variance.
# This is the classical time series benchmark — all neural models
# should be compared against it.
#
# Implementation: statsmodels ARIMA with order selection via AIC.
# For daily data, we use ARIMA(1,1,1) as a standard baseline
# (we avoid auto_arima for speed; p,d,q are chosen based on
# stationarity of SLA time series and Box-Jenkins convention).

if not STATSMODELS_OK:
    print("statsmodels not available. Skipping ARIMA.")
else:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.stattools import adfuller

    arima_results = []
    ARIMA_ORDER = (1, 1, 1)  # (p, d, q) — ARIMA(1,1,1) standard baseline
    CI_ALPHA    = 0.05        # for 95% prediction intervals

    print(f"Training SARIMA{ARIMA_ORDER} across 5 locations...")
    print("Note: ARIMA runs only once per location (not seeded — deterministic model)")

    for loc_name in LOCATIONS:
        d = get_splits(loc_name)
        scaler = d['scaler']
        lat, lon = LOCATIONS[loc_name]

        # Use original (unscaled) training data for ARIMA
        sla_train = d['scaler'].inverse_transform(
            np.concatenate([d['y_tr'], d['y_cal']]).reshape(-1,1)).flatten()
        sla_test  = scaler.inverse_transform(d['y_te'].reshape(-1,1)).flatten()

        # Stationarity test
        adf_stat, adf_p, *_ = adfuller(sla_train, autolag='AIC')
        print(f"\n  {loc_name}: ADF p={adf_p:.4f} ({'stationary' if adf_p < 0.05 else 'non-stationary'})")

        try:
            model = ARIMA(sla_train, order=ARIMA_ORDER)
            res   = model.fit(method_kwargs={"warn_convergence": False})

            # One-step-ahead rolling forecast on test set
            lo_preds, hi_preds, mu_preds = [], [], []
            history = list(sla_train)

            for t in range(len(sla_test)):
                m_fit = ARIMA(history, order=ARIMA_ORDER).fit(
                    method_kwargs={"warn_convergence": False})
                fc    = m_fit.get_forecast(steps=1)
                mu_t  = float(fc.predicted_mean.iloc[0])
                ci    = fc.conf_int(alpha=CI_ALPHA)
                lo_t  = float(ci.iloc[0, 0])
                hi_t  = float(ci.iloc[0, 1])
                mu_preds.append(mu_t); lo_preds.append(lo_t); hi_preds.append(hi_t)
                history.append(sla_test[t])   # update with true observation (rolling)

            lo_m = np.array(lo_preds); hi_m = np.array(hi_preds)
            picp, mpiw, wis, cwc = full_metrics(sla_test, lo_m, hi_m)
            print(f"  ARIMA{ARIMA_ORDER}: PICP={picp:.1f}% | MPIW={mpiw:.5f} | WIS={wis:.5f}")

            arima_results.append(dict(
                location=loc_name, lat=lat, lon=lon, model=f'ARIMA{ARIMA_ORDER}',
                avg_picp=picp, std_picp=0.0,   # deterministic — no std
                avg_mpiw=mpiw, std_mpiw=0.0,
                avg_wis=wis,   std_wis=0.0,
                avg_cwc=cwc,   std_cwc=0.0,
                n_seeds=1, n_train_seqs=len(sla_train), n_test_pts=len(sla_test)
            ))
        except Exception as e:
            print(f"  ARIMA failed for {loc_name}: {e}")

    df_arima = pd.DataFrame(arima_results)
    df_arima.to_csv(os.path.join(SAVE_DIR, 'results_ARIMA_IO.csv'), index=False)
    print(f"\nSaved ARIMA results.")
    print(df_arima[['location','avg_picp','avg_mpiw','avg_cwc']].to_string(index=False))

In [ ]:
# ── Cell 6: Summary plots for new models ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

locs = list(LOCATIONS.keys())
x    = np.arange(len(locs))
w    = 0.25

# PICP
ax = axes[0]
kde_picps   = [float(df_kde[df_kde['location']==l]['avg_picp'].values[0]) if l in df_kde['location'].values else np.nan for l in locs]
mdn_picps   = [float(df_mdn[df_mdn['location']==l]['avg_picp'].values[0]) if l in df_mdn['location'].values else np.nan for l in locs]
arima_picps = [float(df_arima[df_arima['location']==l]['avg_picp'].values[0]) if (STATSMODELS_OK and l in df_arima['location'].values) else np.nan for l in locs]

ax.bar(x-w, kde_picps,   w, label='LSTM-KDE',     color='#3498db', alpha=0.85)
ax.bar(x,   mdn_picps,   w, label='MDN-GRU',      color='#e67e22', alpha=0.85)
ax.bar(x+w, arima_picps, w, label='ARIMA(1,1,1)', color='#27ae60', alpha=0.85)
ax.axhline(95, color='navy', lw=2, linestyle='--', label='Target 95%')
ax.fill_between([-0.5, len(locs)-0.5], 94, 96, alpha=0.1, color='navy')
ax.set_xticks(x); ax.set_xticklabels(locs, rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Avg PICP (%)'); ax.set_title('PICP — New Models', fontweight='bold')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# MPIW
ax2 = axes[1]
kde_mpiws   = [float(df_kde[df_kde['location']==l]['avg_mpiw'].values[0])   if l in df_kde['location'].values   else np.nan for l in locs]
mdn_mpiws   = [float(df_mdn[df_mdn['location']==l]['avg_mpiw'].values[0])   if l in df_mdn['location'].values   else np.nan for l in locs]
arima_mpiws = [float(df_arima[df_arima['location']==l]['avg_mpiw'].values[0]) if (STATSMODELS_OK and l in df_arima['location'].values) else np.nan for l in locs]

ax2.bar(x-w, kde_mpiws, w, label='LSTM-KDE', color='#3498db', alpha=0.85)
ax2.bar(x,   mdn_mpiws, w, label='MDN-GRU',  color='#e67e22', alpha=0.85)
ax2.bar(x+w, arima_mpiws, w, label='ARIMA(1,1,1)', color='#27ae60', alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(locs, rotation=25, ha='right', fontsize=9)
ax2.set_ylabel('Avg MPIW (m)'); ax2.set_title('MPIW — New Models', fontweight='bold')
ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)

# CWC
ax3 = axes[2]
kde_cwcs   = [float(df_kde[df_kde['location']==l]['avg_cwc'].values[0])   if l in df_kde['location'].values   else np.nan for l in locs]
mdn_cwcs   = [float(df_mdn[df_mdn['location']==l]['avg_cwc'].values[0])   if l in df_mdn['location'].values   else np.nan for l in locs]
arima_cwcs = [float(df_arima[df_arima['location']==l]['avg_cwc'].values[0]) if (STATSMODELS_OK and l in df_arima['location'].values) else np.nan for l in locs]

ax3.bar(x-w, kde_cwcs, w, label='LSTM-KDE', color='#3498db', alpha=0.85)
ax3.bar(x,   mdn_cwcs, w, label='MDN-GRU',  color='#e67e22', alpha=0.85)
ax3.bar(x+w, arima_cwcs, w, label='ARIMA(1,1,1)', color='#27ae60', alpha=0.85)
ax3.set_xticks(x); ax3.set_xticklabels(locs, rotation=25, ha='right', fontsize=9)
ax3.set_ylabel('Avg CWC'); ax3.set_title('CWC — New Models\n(lower = better)', fontweight='bold')
ax3.legend(fontsize=8); ax3.grid(axis='y', alpha=0.3)

plt.suptitle('New Probabilistic Forecasting Models — Indian Ocean SLA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'plot_new_models.png'), dpi=140, bbox_inches='tight')
plt.show()
print("All new model results saved.")